# Analisis Fondasi Data Energi HKUST dan Cuaca HKO

Notebook ini menyajikan analisis data untuk topik konsumsi energi kampus menggunakan smart meter HKUST dan data cuaca Hong Kong Observatory. Struktur notebook mengikuti alur OSEMN pada bagian yang sudah memiliki data final: Obtain, Scrub, Explore awal, dan Model.

Fokus analisis saat ini adalah dataset harian T1440 yang bersih, terdokumentasi, memiliki konteks metadata, memiliki fitur turunan untuk analisis, dan menghasilkan deteksi anomali yang siap digunakan sebagai sumber Power BI. Interpretasi rekomendasi akhir tidak disajikan karena membutuhkan matriks insight dan rekomendasi berbasis bukti yang terpisah.

## Setup Analisis

Seluruh tabel yang digunakan di notebook ini berasal dari folder processed proyek. Notebook berperan sebagai ringkasan analitis, sedangkan proses parsing, cleaning, dan pembentukan datamart dilakukan melalui script pipeline.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from config import PROCESSED_DIR, PROFILE_DIR, TARGET_BUILDING, SELECTED_METERS

## O - Obtain: Sumber Data dan Cakupan

Dataset utama berasal dari smart meter HKUST dengan interval harian T1440. Dataset pendukung berasal dari Hong Kong Observatory dan digunakan untuk memberikan konteks cuaca harian.

Cakupan utama analisis diarahkan pada meter aktif di `Cheng_Yu_Tung_Building` karena kumpulan meter ini memiliki coverage penuh, sinyal konsumsi aktif, dan metadata yang dapat dipetakan. Meter lain tetap dicatat dalam inventory agar kualitas data dan keputusan eksklusi dapat ditelusuri.

In [2]:
inventory = pd.read_csv(PROFILE_DIR / "t1440_meter_inventory_with_decision.csv")
weather = pd.read_csv(PROCESSED_DIR / "hko_weather_daily.csv", parse_dates=["date"])
obtain_summary = pd.DataFrame(
    [
        ["Jumlah meter T1440", int(inventory["meter_id"].nunique())],
        ["Jumlah meter subset utama", len(SELECTED_METERS)],
        ["Gedung fokus", TARGET_BUILDING],
        ["Jumlah hari cuaca HKO", len(weather)],
        ["Awal periode cuaca", weather["date"].min().date().isoformat()],
        ["Akhir periode cuaca", weather["date"].max().date().isoformat()],
    ],
    columns=["Metrik", "Nilai"],
)
display(obtain_summary)
display(inventory["quality_flag"].value_counts().rename_axis("quality_flag").reset_index(name="meter_count"))

,Metrik,Nilai
0,Jumlah meter T1440,26
1,Jumlah meter subset utama,12
2,Gedung fokus,Cheng_Yu_Tung_Building
3,Jumlah hari cuaca HKO,878
4,Awal periode cuaca,2022-01-01
5,Akhir periode cuaca,2024-05-27


,quality_flag,meter_count
0,active,18
1,zero_constant,4
2,near_zero_low_signal,3
3,short_coverage,1


## S - Scrub: Pembersihan dan Integrasi Data

Tahap Scrub menormalisasi tanggal, mengubah pembacaan meter menjadi konsumsi harian melalui differencing, menandai kasus kualitas data, dan menggabungkan konsumsi energi dengan cuaca harian berdasarkan tanggal.

Tabel akhir disusun sebagai star schema sederhana agar mudah digunakan di Power BI. Grain utama fact table adalah satu baris per tanggal dan meter terpilih.

In [3]:
dim_date = pd.read_csv(PROCESSED_DIR / "dim_date.csv", parse_dates=["date"])
dim_entity = pd.read_csv(PROCESSED_DIR / "dim_entity.csv")
dim_scenario = pd.read_csv(PROCESSED_DIR / "dim_scenario.csv")
fact = pd.read_csv(PROCESSED_DIR / "fact_energy_weather_daily.csv", parse_dates=["date"])

scrub_summary = pd.DataFrame(
    [
        ["Baris dim_date", len(dim_date)],
        ["Baris dim_entity", len(dim_entity)],
        ["Baris dim_scenario", len(dim_scenario)],
        ["Baris fact_energy_weather_daily", len(fact)],
        ["Jumlah entity di fact", int(fact["entity_id"].nunique())],
        ["Awal periode fact", fact["date"].min().date().isoformat()],
        ["Akhir periode fact", fact["date"].max().date().isoformat()],
        ["Baris model eligible", int(fact["is_model_eligible"].sum())],
    ],
    columns=["Metrik", "Nilai"],
)
display(scrub_summary)

,Metrik,Nilai
0,Baris dim_date,878
1,Baris dim_entity,26
2,Baris dim_scenario,3
3,Baris fact_energy_weather_daily,10524
4,Jumlah entity di fact,12
5,Awal periode fact,2022-01-02
6,Akhir periode fact,2024-05-27
7,Baris model eligible,10524


Validasi berikut memastikan bahwa tabel dimensi memiliki key unik dan fact table tidak memiliki duplikasi kombinasi tanggal dan entity.

In [4]:
checks = {
    "dim_date_unique": not dim_date.duplicated(["date"]).any(),
    "dim_entity_unique": not dim_entity.duplicated(["entity_id"]).any(),
    "dim_scenario_unique": not dim_scenario.duplicated(["scenario"]).any(),
    "fact_date_entity_unique": not fact.duplicated(["date", "entity_id"]).any(),
    "selected_meters_all_present": set(SELECTED_METERS).issubset(set(dim_entity["meter_id"])),
}
display(checks)
assert all(checks.values()), checks

{'dim_date_unique': True,
 'dim_entity_unique': True,
 'dim_scenario_unique': True,
 'fact_date_entity_unique': True,
 'selected_meters_all_present': True}

## E - Explore: Ringkasan Awal Konsumsi dan Kualitas Data

Eksplorasi awal dilakukan untuk membaca skala konsumsi, kontribusi meter, dan status kualitas data. Hasil pada bagian ini belum dimaksudkan sebagai interpretasi akhir, tetapi memberikan dasar yang kuat untuk visualisasi dan modelling berikutnya.

In [5]:
consumption_summary = fact["daily_consumption"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).to_frame("daily_consumption")
display(consumption_summary)

top_meter = (
    fact.groupby(["entity_id", "meter_name"], as_index=False)["daily_consumption"]
    .sum()
    .sort_values("daily_consumption", ascending=False)
)
top_meter["contribution_pct"] = top_meter["daily_consumption"] / top_meter["daily_consumption"].sum()
top_meter["cumulative_contribution_pct"] = top_meter["contribution_pct"].cumsum()
display(top_meter.head(12))

,daily_consumption
count,10524.000000
mean,1300.521570
std,897.735631
min,110.000000
25%,700.000000
50%,980.000000
75%,1980.000000
95%,3160.000000
max,4100.000000


,entity_id,meter_name,daily_consumption,contribution_pct,cumulative_contribution_pct
1,D0849,Meter_D0849,2822400.0,0.206215,0.206215
0,D0848,Meter_D0848,2053420.0,0.150030,0.356245
6,D0854,Meter_D0854,1866900.0,0.136403,0.492648
5,D0853,Meter_D0853,1774050.0,0.129619,0.622267
10,D0862,Meter_D0862,898620.0,0.065656,0.687923
7,D0857,Meter_D0857,877200.0,0.064091,0.752015
3,D0851,Meter_D0851,855730.0,0.062523,0.814537
9,D0861,Meter_D0861,760510.0,0.055566,0.870103
4,D0852,Meter_D0852,679530.0,0.049649,0.919752
8,D0860,Meter_D0860,607590.0,0.044393,0.964145


Ringkasan kualitas data menunjukkan bahwa meter bermasalah tetap terdokumentasi dalam dimensi entity, sedangkan fact utama hanya berisi meter aktif terpilih. Nilai cuaca yang hilang dipertahankan pada kolom asli dan disediakan kolom model-safe untuk kebutuhan analisis lanjutan.

In [6]:
data_quality = pd.read_csv(PROCESSED_DIR / "data_quality_summary.csv")
display(data_quality)
display(fact["data_quality_flag"].value_counts().rename_axis("data_quality_flag").reset_index(name="rows"))

,summary_group,metric,value,notes
0,meter_quality_flag,active,18,Count of T1440 meters by quality flag.
1,meter_quality_flag,near_zero_low_signal,3,Count of T1440 meters by quality flag.
2,meter_quality_flag,short_coverage,1,Count of T1440 meters by quality flag.
3,meter_quality_flag,zero_constant,4,Count of T1440 meters by quality flag.
4,selection_decision,comparison_or_secondary,4,Count of T1440 meters by selection decision.
5,selection_decision,exclude_from_main_model,10,Count of T1440 meters by selection decision.
6,selection_decision,selected_main_subset,12,Count of T1440 meters by selection decision.
7,weather_missing_days,rainfall_mm,5,Missing HKO rainfall days in project period.
8,weather_missing_days,global_solar_radiation_mj_m2,1,Missing HKO solar radiation days in project pe...
9,fact_rows,fact_energy_weather_daily,10524,Rows in final Date x Entity fact table.


,data_quality_flag,rows
0,valid,10452
1,weather_imputed,72


## M - Model: Deteksi Anomali Konsumsi Energi

Tujuan pemodelan adalah menandai observasi tanggal dan meter yang memiliki pola konsumsi tidak biasa dibandingkan riwayat konsumsi entity yang sama serta konteks waktu dan cuaca. Dataset tidak memiliki label resmi normal atau anomali, sehingga pendekatan yang digunakan adalah unsupervised anomaly detection.

Model utama menggunakan Isolation Forest dengan tiga skenario sensitivitas. Baseline IQR dan Z-score digunakan sebagai pembanding agar hasil model tidak berdiri sendiri.

In [7]:
fact_anomaly = pd.read_csv(PROCESSED_DIR / "fact_anomaly_scenarios.csv", parse_dates=["date"])
model_eval = pd.read_csv(PROCESSED_DIR / "model_evaluation_summary.csv")
case_review = pd.read_csv(PROCESSED_DIR / "anomaly_case_review.csv", parse_dates=["date"])
entity_scorecard = pd.read_csv(PROCESSED_DIR / "entity_scorecard.csv")
feature_summary = pd.read_csv(PROCESSED_DIR / "feature_engineering_summary.csv")

model_outputs = pd.DataFrame(
    [
        ["fact_anomaly_scenarios", len(fact_anomaly)],
        ["model_evaluation_summary", len(model_eval)],
        ["anomaly_case_review", len(case_review)],
        ["entity_scorecard", len(entity_scorecard)],
        ["feature_engineering_summary", len(feature_summary)],
    ],
    columns=["Tabel", "Jumlah baris"],
)
display(model_outputs)

,Tabel,Jumlah baris
0,fact_anomaly_scenarios,31260
1,model_evaluation_summary,3
2,anomaly_case_review,20
3,entity_scorecard,12
4,feature_engineering_summary,29


Fitur model menggabungkan konsumsi harian, deviasi rolling 7 hari, lag konsumsi, kalender, dan cuaca model-safe. Kolom kualitas data digunakan untuk filter dan interpretasi, bukan sebagai sinyal utama model.

In [8]:
feature_groups = (
    feature_summary.groupby(["feature_group", "intended_use"], as_index=False)
    .agg(feature_count=("feature_name", "count"), missing_rate_max=("missing_rate", "max"))
    .sort_values(["feature_group", "intended_use"])
)
display(feature_groups)

,feature_group,intended_use,feature_count,missing_rate_max
0,calendar,"dashboard,model",2,0.000000
1,consumption_transform,"eda,interpretation",2,0.000000
2,consumption_transform,"eda,model,interpretation",1,0.000000
3,feature_readiness,"model,dashboard",2,0.000000
4,lag_consumption,"model,interpretation",4,0.007982
5,model_input_existing,model,12,0.000000
6,rolling_consumption,"model,interpretation",4,0.005321
7,weather_context,"eda,dashboard,interpretation",2,0.000000


Ringkasan evaluasi berikut menunjukkan jumlah anomali pada masing-masing skenario. Karena tidak ada ground-truth label, evaluasi dibaca melalui kesesuaian jumlah flag dengan parameter contamination, agreement dengan baseline IQR/Z-score, stabilitas antar skenario, dan tinjauan kasus.

In [9]:
display(
    model_eval[
        [
            "scenario",
            "contamination",
            "eligible_row_count",
            "anomaly_count",
            "anomaly_rate",
            "iqr_agreement_count",
            "zscore_agreement_count",
            "both_baseline_agreement_count",
        ]
    ]
)

,scenario,contamination,eligible_row_count,anomaly_count,anomaly_rate,iqr_agreement_count,zscore_agreement_count,both_baseline_agreement_count
0,strict,0.03,10420,313,0.030038,4,4,4
1,balanced,0.05,10420,521,0.050000,10,6,6
2,sensitive,0.10,10420,1042,0.100000,24,12,12


Skenario balanced digunakan sebagai skenario default untuk peninjauan dashboard. Tabel berikut menampilkan contoh kandidat anomali dengan skor tertinggi, deviasi dari pola konsumsi terkini, serta konteks cuaca dan baseline.

In [10]:
display(
    case_review[
        [
            "date",
            "entity_id",
            "building",
            "scenario",
            "anomaly_rank_within_scenario",
            "daily_consumption",
            "pct_deviation_from_rolling_mean_7d",
            "mean_temperature_c",
            "rainfall_mm_model",
            "baseline_agreement_count",
            "scenario_anomaly_count",
            "consumption_context_label",
            "weather_context_label",
        ]
    ].head(10)
)

,date,entity_id,building,scenario,anomaly_rank_within_scenario,daily_consumption,pct_deviation_from_rolling_mean_7d,mean_temperature_c,rainfall_mm_model,baseline_agreement_count,scenario_anomaly_count,consumption_context_label,weather_context_label
0,2024-05-04,D0849,Cheng_Yu_Tung_Building,balanced,1,4000.0,0.076923,23.5,256.0,0,3,near_recent_pattern,rainy_day
1,2023-07-17,D0849,Cheng_Yu_Tung_Building,balanced,2,2840.0,-0.203526,29.0,15.0,0,3,below_recent_pattern,hot_and_rainy
2,2023-10-09,D0849,Cheng_Yu_Tung_Building,balanced,3,2900.0,-0.139831,24.4,201.0,0,3,near_recent_pattern,rainy_day
3,2023-09-02,D0849,Cheng_Yu_Tung_Building,balanced,4,3300.0,-0.083333,26.4,17.0,0,3,near_recent_pattern,rainy_day
4,2023-10-01,D0849,Cheng_Yu_Tung_Building,balanced,5,3000.0,-0.166667,29.1,0.0,0,3,near_recent_pattern,hot_day
5,2024-04-23,D0849,Cheng_Yu_Tung_Building,balanced,6,4000.0,0.060606,24.5,68.0,0,3,near_recent_pattern,rainy_day
6,2023-07-02,D0849,Cheng_Yu_Tung_Building,balanced,7,3050.0,-0.172801,27.5,0.0,0,3,near_recent_pattern,ordinary_weather_context
7,2023-09-18,D0849,Cheng_Yu_Tung_Building,balanced,8,3000.0,-0.176471,28.5,0.5,0,3,near_recent_pattern,hot_and_rainy
8,2023-09-08,D0849,Cheng_Yu_Tung_Building,balanced,9,3900.0,0.050000,25.9,101.0,0,3,near_recent_pattern,rainy_day
9,2023-10-08,D0849,Cheng_Yu_Tung_Building,balanced,10,3400.0,0.025862,23.6,65.5,0,3,near_recent_pattern,rainy_day


Entity scorecard menggabungkan total konsumsi, tingkat anomali pada skenario balanced, konsumsi puncak, dan kualitas data untuk membantu menentukan prioritas audit energi pada level meter terpilih.

In [11]:
display(
    entity_scorecard[
        [
            "entity_priority_rank",
            "entity_id",
            "building",
            "total_consumption",
            "balanced_anomaly_count",
            "balanced_anomaly_rate",
            "strict_anomaly_count",
            "sensitive_anomaly_count",
            "entity_contribution_pct",
            "entity_priority_score",
        ]
    ].head(12)
)

,entity_priority_rank,entity_id,building,total_consumption,balanced_anomaly_count,balanced_anomaly_rate,strict_anomaly_count,sensitive_anomaly_count,entity_contribution_pct,entity_priority_score
0,1,D0849,Cheng_Yu_Tung_Building,2822400.0,396,0.451539,258,621,0.206215,100.00
1,2,D0848,Cheng_Yu_Tung_Building,2053420.0,59,0.067275,22,204,0.150030,52.57
2,3,D0854,Cheng_Yu_Tung_Building,1866900.0,12,0.013683,7,53,0.136403,46.15
3,4,D0853,Cheng_Yu_Tung_Building,1774050.0,8,0.009122,6,21,0.129619,43.07
4,5,D0862,Cheng_Yu_Tung_Building,898620.0,2,0.002281,1,12,0.065656,17.75
5,6,D0851,Cheng_Yu_Tung_Building,855730.0,2,0.002281,1,10,0.062523,16.52
6,7,D0857,Cheng_Yu_Tung_Building,877200.0,1,0.001140,0,8,0.064091,16.07
7,8,D0861,Cheng_Yu_Tung_Building,760510.0,0,0.000000,0,4,0.055566,12.77
8,9,D0852,Cheng_Yu_Tung_Building,679530.0,1,0.001140,0,7,0.049649,12.46
9,10,D0860,Cheng_Yu_Tung_Building,607590.0,0,0.000000,0,7,0.044393,8.15


Keterbatasan model perlu dibaca secara eksplisit. Model tidak membuktikan penyebab anomali, tidak memiliki label supervised untuk validasi akurasi, dan hanya berlaku pada meter T1440 terpilih. Hasil anomali sebaiknya digunakan sebagai daftar kandidat investigasi, bukan sebagai keputusan operasional tunggal.

## Kesiapan Data untuk Power BI

Tabel berikut merupakan dataset yang dapat diimpor ke Power BI Desktop. Relasi utama menghubungkan fact table ke `dim_date`, `dim_entity`, dan `dim_scenario`.

In [12]:
outputs = [
    "fact_energy_weather_daily.csv",
    "fact_anomaly_scenarios.csv",
    "dim_date.csv",
    "dim_entity.csv",
    "dim_scenario.csv",
    "feature_engineering_summary.csv",
    "model_evaluation_summary.csv",
    "anomaly_case_review.csv",
    "entity_scorecard.csv",
    "data_quality_summary.csv",
    "data_dictionary_energy_dashboard.csv",
]
pd.DataFrame({
    "file": outputs,
    "exists": [(PROCESSED_DIR / name).exists() for name in outputs],
    "path": [str(PROCESSED_DIR / name) for name in outputs],
})

,file,exists,path
0,fact_energy_weather_daily.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...
1,fact_anomaly_scenarios.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...
2,dim_date.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...
3,dim_entity.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...
4,dim_scenario.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...
5,feature_engineering_summary.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...
6,model_evaluation_summary.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...
7,anomaly_case_review.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...
8,entity_scorecard.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...
9,data_quality_summary.csv,True,C:\Users\Khalfani Shaquille\Documents\GitHub\K...


## Catatan Keterbatasan

Dataset T1440 harian hanya mencakup 26 meter, dan fact utama pada notebook ini menggunakan 12 meter aktif dari satu gedung. Karena itu, hasil analisis pada notebook ini sebaiknya dibaca sebagai fondasi analisis meter terpilih, bukan klaim representatif untuk seluruh kampus. Data cuaca digunakan sebagai konteks pendukung dan tidak boleh ditafsirkan sebagai penyebab tunggal perubahan konsumsi.